Before we start, let's import the data science libraries into Python.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score

# Example

We used the dataset called "Advertising.xlsx" in Canvas.

-   TV: Money spent on TV ads for a product (in hundreds of \$).
-   Radio: Money spent on Radio ads for a product (in hundreds of \$).
-   Newspaper: Money spent on Newspaper ads for a product (in hundreds of \$).
-   Sales: Sales generated from the product (\$).
-   200 markets

In [3]:
# Load the data into Python
Ads_data = pd.read_excel('Advertising.xlsx')

# Show the data.
Ads_data

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,9.3
3,151.5,41.3,58.5,18.5
4,180.8,10.8,58.4,12.9
...,...,...,...,...
195,38.2,3.7,13.8,7.6
196,94.2,4.9,8.1,9.7
197,177.0,9.3,6.4,12.8
198,283.6,42.0,66.2,25.5


Now, let's set the predictor and response. In the definition of `X_full`, the double bracket in `[]` is important because it allows us to have a pandas DataFrame as output. This makes it easier to fit the linear regression model with **scikit-learn**.

In [4]:
# Chose the predictor.
X_full = Ads_data.filter(['TV', 'Radio', 'Newspaper'])

# Set the response.
Y_full = Ads_data.filter(['Sales'])

## Create training and validation data

To evaluate the performance of a model on unobserved data, we split the current dataset into a training and a validation dataset. To this end, we use the function `train_test_split()` from **scikit-learn**.

The function has three main inputs. One is a matrix of predictor columns only. The other one is a matrix with the responses only. The last one, called `test_size`, sets the proportion of observations from the full dataset that will go to the validation dataset.

Before we apply the function, let's create the two matrices with predictors and the response. To create the matrix of predictors, we use the function `.drop()` from **pandas**. This function will drop the response variable under study, which is `MEDV`.


In [5]:
X_train, X_valid, Y_train, Y_valid = train_test_split(X_full, Y_full,
                                                      test_size = 0.25,
                                                      random_state = 301655)

The function `train_test_split()` makes a clever partition of the data using the *empirical* distribution of the response. Usually, the proportion of the dataset that goes to the test set is 0.20 or 0.30.

We can compile the full training dataset (with predictor and response values) using the function `.concat()` from **pandas**.  Let's see the training data.

In [6]:
training_data = pd.concat([X_train, Y_train], axis = 1)
training_data

,TV,Radio,Newspaper,Sales
44,25.1,25.7,43.3,8.5
169,284.3,10.6,6.4,15.0
150,280.7,13.9,37.0,16.1
64,131.1,42.8,28.9,18.0
173,168.4,7.1,12.8,11.7
...,...,...,...,...
127,80.2,0.0,9.2,8.8
162,188.4,18.1,25.6,14.9
36,266.9,43.8,5.0,25.4
71,109.8,14.3,31.7,12.4


The argument `axis` of the function indicates whether we want to concatenate by columns (`axis = 0`) or by rows (`axis = 1`). In our case, we want to concatenate by rows so that we attach the response column in `Y_train` to the predictor matrix in `X_train`.

**The training dataset will be used to develop the model.**

Using `.concat()`, we can also compile the full validation dataset:

In [7]:
validation_data = pd.concat([X_valid, Y_valid], axis = 1)
validation_data

,TV,Radio,Newspaper,Sales
12,23.8,35.1,65.9,9.2
43,206.9,8.4,26.4,12.9
107,90.4,0.3,23.2,8.7
35,290.7,4.1,8.5,12.8
199,232.1,8.6,8.7,13.4
110,225.8,8.2,56.5,13.4
91,28.6,1.5,33.0,7.3
104,238.2,34.3,5.3,20.7
190,39.5,41.1,5.8,10.8
38,43.1,26.7,35.1,10.1


**The validation dataset will be used to validate the model's performance on unobserved data.**

## Fit a linear regression model in Python

Let's fit a model to predict sales in terms of TV advertising. The model has the following form:

$$\hat{Y}_i = \hat{\beta}_0 + \hat{\beta}_1 X_{1i} + \hat{\beta}_2 X_{2i} + + \hat{\beta}_3 X_{3i},$$

where $Y_i$ is the predicted sales, and $X_{1i}$, $X_{2i}$, and $X_{3i}$ are the TV, radio, and newspaper advertising values, respectively, in the $i$-th marketin the training dataset, $i = 1, \ldots, n_{train}$. Moreover, $\beta_0$ is the intercept and $\beta_1$, $\beta_2$, and $\beta_3$ are coefficients of the TV, radio, and newspaper.

To find the values of the coefficients $\beta_0$, $\beta_1$, $\beta_2$, and $\beta_3$, we use the `LinearRegression()` and `fit()` functions from the **scikit-learn** as follows:

In [8]:
# 1. Create the linear regression model
LRmodel = LinearRegression()

# 2. Fit the model.
LRmodel.fit(X_train, Y_train)

LinearRegression()

The following commands allow you to show the estimated coefficients of the model.

In [9]:
print("Coefficients:", LRmodel.coef_)

Coefficients: [[ 0.04620676  0.18682436 -0.0059342 ]]


We can also show the estimated intercept.

In [10]:
print("Intercept:", LRmodel.intercept_)

Intercept: [3.25814952]


The estimated model thus is

$$\hat{Y}_i = 3.258 + 0.046 X_{i1} + 0.186 X_{i2} - 0.005X_{i3}.$$

- The average sales are 3.258 thousands of dollars when the advertising budgets for TV, Radio, and Newspaper are 0 dollars.

- Increasing the advertising TV by 1 hundreds of dollars increases the average sales by 0.046 thousands of dollars.  

- Increasing the advertising Radio by 1 hundreds of dollars increases the average sales by 0.186 thousands of dollars.  

- Increasing the advertising Newspaper by 1 hundreds of dollars reduces the average sales by 0.005 thousands of dollars.


## Evaluating the Predictive Performance

To evaluate the model's performance, we use the validation dataset. Specifically, we use the predictor matrix stored in `X_valid`.

In Python, we make the prediction using the pre-trained `LRmodel`.

In [11]:
Y_pred = LRmodel.predict(X_valid)
Y_pred

array([[10.52434137],
       [14.23098934],
       [ 7.3536142 ],
       [17.40599313],
       [15.5377999 ],
       [14.88831266],
       [ 4.66407062],
       [20.64122346],
       [12.72737925],
       [10.02958064],
       [10.63711132],
       [ 8.59954734],
       [16.33261478],
       [14.88553778],
       [ 9.14044726],
       [14.51989702],
       [14.06629238],
       [14.48250904],
       [13.13623327],
       [19.40537834],
       [12.57582429],
       [15.6645823 ],
       [ 9.91960088],
       [19.97845115],
       [16.59368035],
       [ 9.26742087],
       [12.55224007],
       [14.42815609],
       [15.66772459],
       [ 8.38222153],
       [16.39344346],
       [11.67275046],
       [13.67007014],
       [23.32784723],
       [23.81616484],
       [13.0074783 ],
       [ 6.74362231],
       [21.32174479],
       [11.66665686],
       [14.20058045],
       [19.30002745],
       [18.78675738],
       [ 7.81809014],
       [14.22024009],
       [16.80900785],
       [15

In [12]:
X_valid

,TV,Radio,Newspaper
12,23.8,35.1,65.9
43,206.9,8.4,26.4
107,90.4,0.3,23.2
35,290.7,4.1,8.5
199,232.1,8.6,8.7
110,225.8,8.2,56.5
91,28.6,1.5,33.0
104,238.2,34.3,5.3
190,39.5,41.1,5.8
38,43.1,26.7,35.1


In [13]:
Y_valid

,Sales
12,9.2
43,12.9
107,8.7
35,12.8
199,13.4
110,13.4
91,7.3
104,20.7
190,10.8
38,10.1


To evaluate the model, we use the mean squared error in the Python `mean_squared_error()` function. Recall that the responses from the validation dataset are in `Y_valid`, and the model predictions are in `Y_pred`.

In [14]:
mse = mean_squared_error(Y_valid, Y_pred)  # Mean Squared Error (MSE)
print(f"Mean Squared Error: {mse:.2f}")

Mean Squared Error: 5.09


To obtain the **root mean squared error (RMSE)**, we use the `root_mean_squared_error()` function instead.

In [15]:
rmse = root_mean_squared_error(Y_valid, Y_pred)
print(f"Root Mean Squared Error: {rmse:.2f}")

Root Mean Squared Error: 2.26


In the context of Data Science, we have another important metric called $R^2$, which can be interpreted as the *squared* correlation between the actual responses and those predicted by the model. The higher the correlation, the better the agreement between the predicted and actual responses.

We compute $R^2$ in Python as follows:

In [16]:
rtwo_sc = r2_score(Y_valid, Y_pred)  # Rsquared
print(round(rtwo_sc, 2))

0.78


# K-fold cross validation

K-fold cross validation (CV) is a popular technique to better estimate the predictive performance of a supervised model or algorithm. The basic idea is to divide the **full** data into $K$ equally-sized partitions or *folds*. So, instead of having a training and validation dataset, K-fold CV creates several partitions of the full data set into training and validation. In this way, it leverages the entire dataset, as each observation is used for training and validation.

To apply K-fold CV, we use the function cross_val_score from **scikit-learn**

In [17]:
# 1. Define a new linear regression model
LRmodel_cv = LinearRegression()

# 2. Apply 5-fold CV
neg_cv_MSEs = cross_val_score(LRmodel_cv, X_full, Y_full, cv = 5,
                        scoring = "neg_mean_squared_error")

# 3. Show scores
print(neg_cv_MSEs)

[-3.1365399  -2.42566776 -1.58522508 -5.42615506 -2.79114519]


Note that we are using the full data set in `X_full` and `Y_full` above.

Unfortunately, `cross_val_score()` outputs negative scores. We simply turn them into positive by multiplying them by -1 or adding a `-` symbol.

In [18]:
cv_MSEs = -neg_cv_MSEs

After that, we average the values using `.mean()` to obtain a the $5$-fold CV estimate.

In [19]:
MSE_cv = cv_MSEs.mean()
print(round(MSE_cv, 3))

3.073


K-fold CV can be used in terms of any predictive performance metric. For example, we can compute a $K$-fold CV estimate for any evaluation metric including the $R^2$. To this end, we set `scoring = "r2"`.

In [20]:
# 1. Define a new linear regression model
LRmodel_cv = LinearRegression()

# 2. Apply 5-fold CV
cv_Rsq = cross_val_score(LRmodel_cv, X_full, Y_full, cv = 5,
                        scoring = "r2")

# 3. Compute CV estimate
Rsq_cv = cv_Rsq.mean()
print(round(Rsq_cv, 3))

0.887


Or, we can compute a $K$-fold CV estimate for the RMSE.

In [21]:
LRmodel_cv = LinearRegression()
neg_cv_RMSEs = cross_val_score(LRmodel_cv, X_full, Y_full, cv = 5,
                        scoring = "neg_root_mean_squared_error")
cv_RMSEs = -neg_cv_RMSEs
RMSE_cv = cv_RMSEs.mean()
print(round(RMSE_cv, 3))

1.718
